# Lung Cancer Classification — AAO + EfficientNetB0
**Dataset:** IQ-OTHNCCD Lung Cancer Dataset  
**Optimizer:** Adaptive Aquila Optimizer (AAO) from mealpy  
**Deep Model:** EfficientNetB0 (Transfer Learning)

## 1. Mount Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/My Drive/Collab Files/P29Sach6')
%cd /content/drive/My\ Drive/Collab\ Files/P29Sach6

## 2. Install Dependencies

In [ ]:
!pip install mealpy==2.5.1 pywavelets imbalanced-learn scikit-image -q

## 3. Import Packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2, os, random, pywt
from collections import Counter
from PIL import Image
from skimage import exposure

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, BaggingClassifier
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                             f1_score, mean_squared_error, mean_absolute_error,
                             r2_score, roc_auc_score, roc_curve,
                             confusion_matrix, classification_report)
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from mealpy.swarm_based.AO import AdaptiveAO   # Adaptive Aquila Optimizer

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 4. Dataset Configuration

In [ ]:
DIRECTORY = r'Data/The IQ-OTHNCCD lung cancer dataset/The IQ-OTHNCCD lung cancer dataset'
CATEGORIES = ['Bengin cases', 'Malignant cases', 'Normal cases']
IMG_SIZE = 224   # EfficientNet default input size
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 5. Exploratory Data Analysis — Class Distribution & Image Sizes

In [ ]:
# Count images per class
class_counts = {cat: len(os.listdir(os.path.join(DIRECTORY, cat))) for cat in CATEGORIES}
print('Class Counts:', class_counts)

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].bar(class_counts.keys(), class_counts.values(), color=['steelblue','tomato','seagreen'])
ax[0].set_title('Class Distribution')
ax[0].set_ylabel('Count')

ax[1].pie(class_counts.values(), labels=class_counts.keys(),
          autopct='%1.1f%%', colors=['steelblue','tomato','seagreen'])
ax[1].set_title('Class Proportion')
plt.tight_layout()
plt.show()

## 6. Sample Image Visualization per Class

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('Sample Images per Class', fontsize=14, fontweight='bold')

for r, cat in enumerate(CATEGORIES):
    path = os.path.join(DIRECTORY, cat)
    files = os.listdir(path)[:4]
    for c, f in enumerate(files):
        img = cv2.imread(os.path.join(path, f), 0)
        axes[r, c].imshow(img, cmap='gray')
        axes[r, c].set_title(cat.split()[0], fontsize=9)
        axes[r, c].axis('off')

plt.tight_layout()
plt.show()

## 7. Data Loading

In [ ]:
data = []
for i, cat in enumerate(CATEGORIES):
    path = os.path.join(DIRECTORY, cat)
    for f in os.listdir(path):
        img = cv2.imread(os.path.join(path, f), 0)
        if img is not None:
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            data.append([img, i])

random.shuffle(data)
X_raw = np.array([d[0] for d in data])
y = np.array([d[1] for d in data])

print(f'Total images: {len(X_raw)} | Shape: {X_raw.shape}')
print('Class distribution:', Counter(y))

## 8. Preprocessing Pipeline with Step-by-Step Visualization

In [ ]:
# ── Step 1: Normalize ──────────────────────────────────────────────────────────
X_norm = X_raw / 255.0

# ── Step 2: Outlier Removal (Isolation Forest) ────────────────────────────────
clf_iso = IsolationForest(contamination=0.05, random_state=SEED)
mask = clf_iso.fit_predict(X_norm.reshape(len(X_norm), -1))
X_clean = X_norm[mask == 1]
y_clean = y[mask == 1]
print(f'After outlier removal: {len(X_clean)} images (removed {(mask==-1).sum()})')

# ── Step 3: Gaussian Blur (noise suppression) ─────────────────────────────────
X_blur = np.array([cv2.GaussianBlur(img.astype('float32'), (3, 3), 0) for img in X_clean])

# ── Step 4: CLAHE — Contrast Limited Adaptive Histogram Equalization ──────────
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
X_clahe = np.array([
    clahe.apply((img * 255).astype(np.uint8)) / 255.0
    for img in X_blur
])

# ── Step 5: Wavelet Denoising (LL sub-band) ───────────────────────────────────
def wavelet_denoise(img):
    LL, _ = pywt.dwt2(img, 'bior1.3')
    return cv2.resize(LL, (IMG_SIZE, IMG_SIZE))

X_wave = np.array([wavelet_denoise(img) for img in X_clahe])
# Re-normalize after wavelet
X_wave = (X_wave - X_wave.min()) / (X_wave.max() - X_wave.min() + 1e-8)

print('Preprocessing complete. Final shape:', X_wave.shape)

In [ ]:
# Visualize each preprocessing step for one sample
idx = 0
steps = [
    (X_raw[idx], 'Original'),
    (X_clean[idx], 'Normalized'),
    (X_blur[idx], 'Gaussian Blur'),
    (X_clahe[idx], 'CLAHE'),
    (X_wave[idx], 'Wavelet LL'),
]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('Preprocessing Pipeline (Sample Image)', fontsize=13, fontweight='bold')
for ax, (img, title) in zip(axes, steps):
    ax.imshow(img, cmap='gray')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 9. Train/Test Split & SMOTE Oversampling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_wave, y_clean, test_size=0.25, random_state=SEED, stratify=y_clean)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print('Train distribution:', Counter(y_train))

# SMOTE
X_flat = X_train.reshape(len(X_train), -1)
smote = SMOTE(random_state=SEED)
X_sm, y_sm = smote.fit_resample(X_flat, y_train)
X_sm = X_sm.reshape(-1, IMG_SIZE, IMG_SIZE)
print('After SMOTE:', Counter(y_sm))

# Visualize SMOTE effect
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (counts, title) in zip(axes, [
    (Counter(y_train), 'Before SMOTE'),
    (Counter(y_sm),    'After SMOTE')
]):
    ax.bar([CATEGORIES[k].split()[0] for k in sorted(counts)],
           [counts[k] for k in sorted(counts)],
           color=['steelblue','tomato','seagreen'])
    ax.set_title(title)
plt.tight_layout()
plt.show()

## 10. Baseline Classical Models

In [ ]:
X_tr_flat = X_sm.reshape(len(X_sm), -1)
X_te_flat = X_test.reshape(len(X_test), -1)

model_performance = pd.DataFrame(
    columns=['Accuracy','Recall','Precision','F1-Score','MSE','MAE','R2','ROC-AUC'])

def evaluate(name, y_true, y_pred):
    yth = label_binarize(y_true, classes=[0,1,2])
    yph = label_binarize(y_pred, classes=[0,1,2])
    model_performance.loc[name] = [
        accuracy_score(y_true, y_pred),
        recall_score(y_true, y_pred, average='weighted'),
        precision_score(y_true, y_pred, average='weighted', zero_division=0),
        f1_score(y_true, y_pred, average='weighted'),
        mean_squared_error(y_true, y_pred),
        mean_absolute_error(y_true, y_pred),
        r2_score(y_true, y_pred),
        roc_auc_score(yth, yph, average='macro'),
    ]
    print(f'\n── {name} ──')
    print(classification_report(y_true, y_pred, target_names=[c.split()[0] for c in CATEGORIES]))

def plot_cm_roc(name, y_true, y_pred):
    yth = label_binarize(y_true, classes=[0,1,2])
    yph = label_binarize(y_pred, classes=[0,1,2])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    # Confusion Matrix
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', ax=axes[0],
                xticklabels=[c.split()[0] for c in CATEGORIES],
                yticklabels=[c.split()[0] for c in CATEGORIES])
    axes[0].set_title(f'{name} — Confusion Matrix')
    # ROC
    for i, c in enumerate(CATEGORIES):
        fpr, tpr, _ = roc_curve(yth[:, i], yph[:, i])
        auc = roc_auc_score(yth[:, i], yph[:, i])
        axes[1].plot(fpr, tpr, label=f'{c.split()[0]} (AUC={auc:.2f})')
    axes[1].plot([0,1],[0,1],'--', color='gray')
    axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[1].set_title(f'{name} — ROC Curve')
    axes[1].legend()
    plt.tight_layout(); plt.show()

# KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_tr_flat, y_sm)
evaluate('KNN', y_test, knn.predict(X_te_flat))
plot_cm_roc('KNN', y_test, knn.predict(X_te_flat))

# Decision Tree
dt = DecisionTreeClassifier(random_state=SEED)
dt.fit(X_tr_flat, y_sm)
evaluate('DecisionTree', y_test, dt.predict(X_te_flat))
plot_cm_roc('DecisionTree', y_test, dt.predict(X_te_flat))

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=SEED)
rf.fit(X_tr_flat, y_sm)
evaluate('RandomForest', y_test, rf.predict(X_te_flat))
plot_cm_roc('RandomForest', y_test, rf.predict(X_te_flat))

# SVM
svm = SVC(kernel='rbf', C=1)
svm.fit(X_tr_flat, y_sm)
evaluate('SVM', y_test, svm.predict(X_te_flat))
plot_cm_roc('SVM', y_test, svm.predict(X_te_flat))

## 11. Adaptive Aquila Optimizer (AAO) — Hyperparameter Tuning
AAO tunes **learning rate** and **dropout rate** for EfficientNetB0.

In [ ]:
# Convert grayscale → RGB for EfficientNet
X_tr_rgb = np.array([cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_GRAY2RGB) for img in X_sm])
X_te_rgb = np.array([cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_GRAY2RGB) for img in X_test])
print('RGB shapes →', X_tr_rgb.shape, X_te_rgb.shape)

In [ ]:
def build_efficientnet(lr=1e-4, dropout=0.4):
    base = EfficientNetB0(weights='imagenet', include_top=False,
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False          # Freeze backbone (fine-tuning phase 1)
    model = Sequential([
        base,
        GlobalAveragePooling2D(),
        Dropout(dropout),
        Dense(128, activation='relu'),
        Dropout(dropout / 2),
        Dense(3, activation='softmax')
    ])
    model.compile(optimizer=Adam(lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Objective: minimize validation loss using a small proxy subset
PROXY_N   = 200   # quick proxy eval (increase if compute allows)
PROXY_VAL = 50

def aao_objective(solution):
    lr      = 10 ** solution[0]     # log-scale: [-5, -2]
    dropout = solution[1]           # [0.2, 0.6]
    epochs  = int(solution[2])      # [3, 15]
    m = build_efficientnet(lr=lr, dropout=dropout)
    es = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
    hist = m.fit(
        X_tr_rgb[:PROXY_N], y_sm[:PROXY_N],
        validation_data=(X_te_rgb[:PROXY_VAL], y_test[:PROXY_VAL]),
        epochs=epochs, batch_size=16, callbacks=[es], verbose=0
    )
    return min(hist.history['val_loss'])   # minimize val loss

problem_dict = {
    "fit_func": aao_objective,
    "lb": [-5, 0.2, 3],
    "ub": [-2, 0.6, 15],
    "minmax": "min",
    "log_to": "console"
}

EPOCH    = 10     # AAO iterations (increase for better tuning)
POP_SIZE = 5      # agents (increase if compute allows)

aao = AdaptiveAO(epoch=EPOCH, pop_size=POP_SIZE)
best_position, best_fitness = aao.solve(problem_dict)

best_lr      = 10 ** best_position[0]
best_dropout = best_position[1]
best_epochs  = int(best_position[2])

print(f'\n✅ Best LR: {best_lr:.2e} | Dropout: {best_dropout:.3f} | Epochs: {best_epochs}')
print(f'   Best Val Loss: {best_fitness:.4f}')

In [ ]:
# AAO convergence plot
plt.figure(figsize=(8, 4))
plt.plot(aao.history.list_global_best_fit, marker='o', color='royalblue', label='Global Best')
plt.plot(aao.history.list_current_best_fit, alpha=0.5, color='orange', label='Current Best')
plt.xlabel('Iteration')
plt.ylabel('Validation Loss (Objective)')
plt.title('AAO Convergence Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Train Final EfficientNetB0 with AAO Hyperparameters

In [ ]:
final_model = build_efficientnet(lr=best_lr, dropout=best_dropout)
final_model.summary()

es_final = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

history = final_model.fit(
    X_tr_rgb, y_sm,
    validation_data=(X_te_rgb, y_test),
    epochs=best_epochs,
    batch_size=32,
    callbacks=[es_final]
)

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, metric, title in zip(axes, ['accuracy', 'loss'], ['Accuracy', 'Loss']):
    ax.plot(history.history[metric], label='Train')
    ax.plot(history.history[f'val_{metric}'], label='Validation')
    ax.set_title(f'EfficientNetB0 — {title}')
    ax.set_xlabel('Epoch'); ax.set_ylabel(title)
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('AAO-Tuned EfficientNetB0 Training History', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 13. Evaluate EfficientNetB0

In [ ]:
y_pred_prob = final_model.predict(X_te_rgb)
y_pred_eff  = np.argmax(y_pred_prob, axis=1)

evaluate('AAO+EfficientNetB0', y_test, y_pred_eff)
plot_cm_roc('AAO+EfficientNetB0', y_test, y_pred_eff)

## 14. Fine-Tuning — Unfreeze Top Layers (Phase 2)

In [ ]:
# Unfreeze top 20 layers of EfficientNet for fine-tuning
base_layer = final_model.layers[0]
for layer in base_layer.layers[-20:]:
    layer.trainable = True

final_model.compile(
    optimizer=Adam(best_lr / 10),   # Lower LR for fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

es_ft = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)
history_ft = final_model.fit(
    X_tr_rgb, y_sm,
    validation_data=(X_te_rgb, y_test),
    epochs=10,
    batch_size=32,
    callbacks=[es_ft]
)

# Re-evaluate after fine-tuning
y_pred_ft = np.argmax(final_model.predict(X_te_rgb), axis=1)
evaluate('AAO+EfficientNetB0 (Fine-Tuned)', y_test, y_pred_ft)
plot_cm_roc('AAO+EfficientNetB0 (Fine-Tuned)', y_test, y_pred_ft)

## 15. Final Model Comparison

In [ ]:
print('\n======== Model Performance Summary ========')
display(model_performance.sort_values('Accuracy', ascending=False).style
    .background_gradient(cmap='RdYlGn')
    .format('{:.4f}')
    .set_caption('Model Comparison — All Metrics')
)

In [ ]:
# Bar chart comparison
metrics = ['Accuracy', 'F1-Score', 'ROC-AUC']
mp = model_performance[metrics]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = plt.cm.Set2.colors

for ax, metric in zip(axes, metrics):
    bars = ax.barh(mp.index, mp[metric], color=colors[:len(mp)])
    ax.set_xlim(0, 1.05)
    ax.set_xlabel(metric)
    ax.set_title(f'Models by {metric}')
    ax.bar_label(bars, fmt='%.3f', padding=3)

plt.suptitle('Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 16. Save Models

In [ ]:
from joblib import dump
dump(knn, 'knn_model.joblib')
dump(dt,  'decision_tree_model.joblib')
dump(rf,  'random_forest_model.joblib')
dump(svm, 'svm_model.joblib')
final_model.save('aao_efficientnetb0_lungcancer.h5')
print('✅ All models saved.')